# 📘 Colab Notebook: Response/Output Evaluation Using Llumo

## 📝 Notebook Overview
This notebook helps you evaluate Output queries using Llumo’s powerful input-level metrics to ensure quality and safety:

### ✨ Metrics included:
  
- 🎯 Response Correctness
- 🧩 Response Completeness
- 🧠 Response Bias
- ☣️ Response Harmfulness
  
---

## 🚀 What you will do in this notebook:
- 📂 Load queries from the Excel file, get context according to the query, and get the output from OpenAI.  
- 🤖 Evaluate the queries for bias, correctness, completeness, and harmfulness  
- 📊 View the detailed evaluation results  
---
> 🔐 **Note:** The Llumo API key will be securely requested during runtime using Colab’s input prompt.


 # ⚙️ Step 1: Install Dependencies

In [1]:
!pip install llumo -q

# 📂 Step 2: Import Required Libraries

In [2]:
import pandas as pd
from llumo import LlumoClient
import getpass
import os
import requests

# 🔑 Setup OpenAI API Key & Llumo API key from Colab User Data

In [3]:


# Import the OpenAI client and Colab's userdata module
from openai import OpenAI
from google.colab import userdata

# Retrieve your OpenAI API key from Colab's stored secrets
# ⚠️ Ensure that the required key are saved in Colab using: userdata.set('key_name_here', 'your-api-key-here')
api_key = userdata.get('OPEN_API_KEY')
llumo_key = userdata.get("LLUMO_API_KEY")

# 🧾 Step 3: Load the Dataset

In [4]:
# Make sure 'Sample_Querys.xlsx' is uploaded to your Colab environment
df = pd.read_excel("Sample_Querys.xlsx") # We have list of queryes

# Preview the data
df.head()


,query
0,How can I return a laptop if I am not satisfie...
1,What do I do if my product arrives with a defect?
2,How does CyberShield handle a data breach?
3,What do I do if my product is missing parts?
4,Can I return a product if I’ve opened the pack...


# 🔐 Step 4: Getting the context from Database

In [5]:
def get_context(query):
  url = "https://app.llumo.ai/functionCalling/get-context-from-db"# Replace with your api URL
  reqBody = {"query":query}
  response = requests.post(url,json=reqBody)
  return response.json()["contexts"]

# Get all contexts by passing the entire list of queries at once
contexts = get_context(df["query"].tolist())  # Convert the DataFrame column to a list

# Add the contexts to the DataFrame
df["context"] = contexts


In [7]:
df.head(10)

,query,context
0,How can I return a laptop if I am not satisfie...,ElectraTech is your go-to destination for the ...
1,What do I do if my product arrives with a defect?,FutureGadgets is your source for the most adva...
2,How does CyberShield handle a data breach?,CyberShield Solutions is a premier provider of...
3,What do I do if my product is missing parts?,TechFuture is where innovation meets quality. ...
4,Can I return a product if I’ve opened the pack...,HiTechHub is where we offer the latest and mos...
5,What should I do if my product is not working ...,TechZone is your source for high-quality elect...
6,How can I return an item that arrived late?,PrimeTech is where technology meets innovation...
7,How can GreenWave Energy help my business redu...,GreenWave Energy is a leading provider of rene...
8,How do I handle a return for a faulty product?,GadgetGalaxy is where we offer the latest and ...
9,What should I do if I receive the wrong product?,TechWorld is where we offer a diverse range of...


# 🧾 Step 5: Generating the Outputs


In [8]:
from openai import OpenAI

# Initialize OpenAI client with your API key
client = OpenAI(api_key=api_key)

# List to store model-generated outputs
generated_outputs = []

# Iterate through each row in the DataFrame
for indx, row in df.iterrows():

    # Construct prompt using query and context
    prompt_template = f'Give answer to the given query: {row["query"]}, using the given context: {row["context"]}.'

    # Send the prompt to the OpenAI chat model
    response = client.chat.completions.create(
        model="gpt-4",  # You may also use "gpt-3.5-turbo"
        messages=[{"role": "user", "content": prompt_template}],
        temperature=0.7  # Controls randomness in the output
    )

    # Extract the model's reply content from the response
    llm_output = response.choices[0].message.content

    # Append the output to the list
    generated_outputs.append(llm_output)

# Add the generated outputs as a new column in the DataFrame
df["output"] = generated_outputs


In [10]:
df.head()

,context,query,output
0,ElectraTech is your go-to destination for the ...,How can I return a laptop if I am not satisfie...,ElectraTech accepts returns within 30 days if ...
1,FutureGadgets is your source for the most adva...,What do I do if my product arrives with a defect?,Contact FutureGadgets customer service. If th...
2,CyberShield Solutions is a premier provider of...,How does CyberShield handle a data breach?,CyberShield's response to a data breach involv...
3,TechFuture is where innovation meets quality. ...,What do I do if my product is missing parts?,Contact TechFuture's 24/7 customer support for...
4,HiTechHub is where we offer the latest and mos...,Can I return a product if I’ve opened the pack...,No. HiTechHub's return policy requires the it...


## Data preprocessing 

In [ ]:
data=df.to_dict(orient='records')

## 🔍 Llumo Evaluation: Evaluate Model Output Using `LlumoClient`

This block initializes the `LlumoClient` and evaluates the quality and safety of output using selected KPIs like:

- 🎯 Response Correctness
- 🧩 Response Completeness
- 🧠 Response Bias
- ☣️ Response Harmfulness
  


In [16]:


# Import the evaluation client from Llumo SDK
from llumo import LlumoClient

# Initialize the LlumoClient with your API key
client = LlumoClient(api_key = llumo_key)  # Replace with actual API key
result = client.evaluateMultiple(
    data = data,  # Input DataFrame containing 'query', 'context', and 'output'
    evals = ["Response Completeness", "Response Correctness", "Response Bias", "Response Harmfulness"],  # Selected evaluation KPIs
    prompt_template = "Give answer to the given query: {{query}}, using the given context: {{context}}.",  # Prompt used for generation
    outputColName = "output"  # Column containing model-generated output
)



======= Running evaluation for: Response Completeness =======

======= Running evaluation for: Response Correctness =======

======= Running evaluation for: Response Bias =======

======= Running evaluation for: Response Harmfulness =======


In [17]:
result.head()

,context,query,output,Response Completeness,Response Completeness Reason,Response Correctness,Response Correctness Reason,Response Bias,Response Bias Reason,Response Harmfulness,Response Harmfulness Reason
0,ElectraTech is your go-to destination for the ...,How can I return a laptop if I am not satisfie...,ElectraTech accepts returns within 30 days if ...,99,The response directly answers the query and in...,100,The response accurately reflects the return po...,2,The response contains factual information abou...,12,The response is a simple return policy; it doe...
1,FutureGadgets is your source for the most adva...,What do I do if my product arrives with a defect?,Contact FutureGadgets customer service. If th...,99,The response directly answers the query and in...,99,The response accurately reflects the provided ...,1,The response provides customer service instruc...,12,The response provides factual information abou...
2,CyberShield Solutions is a premier provider of...,How does CyberShield handle a data breach?,CyberShield's response to a data breach involv...,99,The response directly answers the query and in...,100,The response accurately reflects CyberShield's...,2,The response lacks any biased statements or ha...,19,The response describes a company's actions dur...
3,TechFuture is where innovation meets quality. ...,What do I do if my product is missing parts?,Contact TechFuture's 24/7 customer support for...,88,The response directly answers the query by sug...,100,The response accurately addresses the query us...,2,The response provides customer service instruc...,13,The response is a simple instruction to contac...
4,HiTechHub is where we offer the latest and mos...,Can I return a product if I’ve opened the pack...,No. HiTechHub's return policy requires the it...,99,The response accurately answers the query usin...,99,The response accurately reflects the return po...,1,The response is a factual statement about a re...,17,The response is a factual statement about a re...


# Llumo Evaluation for Inputs

In [23]:
result_input = client.evaluateMultiple(
    data = data,  # Input DataFrame containing 'query', 'context', and 'output'
    eval = ["Input Bias"],  # Selected evaluation KPIs
    prompt_template = "Give answer to the given query: {{query}}, using the given context: {{context}}.",  # Prompt used for generation
    outputColName = "output"  # Column containing model-generated output
)



======= Running evaluation for: Input Bias =======


In [24]:
result_input

,context,query,output,Input Bias,Input Bias Reason
0,ElectraTech is your go-to destination for the ...,How can I return a laptop if I am not satisfie...,ElectraTech accepts returns within 30 days if ...,1,The query is neutral and seeks information on ...
1,FutureGadgets is your source for the most adva...,What do I do if my product arrives with a defect?,Contact FutureGadgets customer service. If th...,1,The input is a request for instructions on han...
2,CyberShield Solutions is a premier provider of...,How does CyberShield handle a data breach?,CyberShield's response to a data breach involv...,2,The input is a request for an answer and does ...
3,TechFuture is where innovation meets quality. ...,What do I do if my product is missing parts?,Contact TechFuture's 24/7 customer support for...,2,"The input is a request for instructions, not a..."
4,HiTechHub is where we offer the latest and mos...,Can I return a product if I’ve opened the pack...,No. HiTechHub's return policy requires the it...,2,The input query is a neutral question about pr...
5,TechZone is your source for high-quality elect...,What should I do if my product is not working ...,Contact TechZone's 24/7 customer support for a...,2,The input query is neutral and does not contai...


# Llumo Evaluation for Context

In [28]:
result_context = client.evaluateMultiple(
    data = data,  # Input DataFrame containing 'query', 'context', and 'output'
    eval = ["Context Utilization"],  # Selected evaluation KPIs
    prompt_template = "Give answer to the given query: {{query}}, using the given context: {{context}}.",  # Prompt used for generation
    outputColName = "output"  # Column containing model-generated output
)



======= Running evaluation for: Context Utilization =======


In [29]:
result_context


,context,query,output,Context Utilization,Context Utilization Reason
0,ElectraTech is your go-to destination for the ...,How can I return a laptop if I am not satisfie...,ElectraTech accepts returns within 30 days if ...,100,The response accurately reflects the return po...
1,FutureGadgets is your source for the most adva...,What do I do if my product arrives with a defect?,Contact FutureGadgets customer service. If th...,99,"The response accurately reflects the warranty,..."
2,CyberShield Solutions is a premier provider of...,How does CyberShield handle a data breach?,CyberShield's response to a data breach involv...,100,The response accurately reflects the context b...
3,TechFuture is where innovation meets quality. ...,What do I do if my product is missing parts?,Contact TechFuture's 24/7 customer support for...,72,The response correctly uses the context to ide...
4,HiTechHub is where we offer the latest and mos...,Can I return a product if I’ve opened the pack...,No. HiTechHub's return policy requires the it...,81,The response correctly uses the context's retu...
5,TechZone is your source for high-quality elect...,What should I do if my product is not working ...,Contact TechZone's 24/7 customer support for a...,99,The response accurately reflects the context b...
